# 01 — Fetch ML/AI repositories from GitHub Search API

Pulls repos tagged with ML/AI topics, deduplicates, saves raw JSON.

In [ ]:
import requests
import json
import time
import os
from dotenv import load_dotenv

load_dotenv()
TOKEN = os.getenv("GITHUB_TOKEN")
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"}
BASE_URL = "https://api.github.com" 

In [ ]:
def search_repos(topic, max_pages=10):
    repos = []
    for page in range(1, max_pages + 1):
        url = f"{BASE_URL}/search/repositories"
        params = {
            "q": f"topic:{topic} stars:>100",
            "sort": "stars",
            "order": "desc",
            "per_page": 100,
            "page": page,
        }
        r = requests.get(url, headers=HEADERS, params=params)
        if r.status_code != 200:
            print(f"Error on {topic} page {page}: {r.status_code}")
            break
        items = r.json()["items"]
        repos.extend(items)
        if len(items) < 100:
            break
        time.sleep(2)
    print(f"{topic}: collected {len(repos)} repos")
    return repos

In [ ]:
TOPICS = ["machine-learning", "deep-learning", "neural-network", "artificial-intelligence"]

all_repos = []
for topic in TOPICS:
    all_repos.extend(search_repos(topic))

print(f"\nTotal before dedup: {len(all_repos)}")

In [ ]:
unique = {}
for r in all_repos:
    unique[r["full_name"]] = r

deduped = list(unique.values())
print(f"After dedup: {len(deduped)}")

In [ ]:
with open("../data/raw/repos_raw.json", "w") as f:
    json.dump(deduped, f)
print("Saved to data/raw/repos_raw.json")

In [ ]:
deduped.sort(key=lambda x: x["stargazers_count"], reverse=True)
print("Top 5 by stars:")
for r in deduped[:5]:
    print(f"  {r['full_name']}: {r['stargazers_count']:,}")